# Course Benchmark: FD Acoustic 1D Animation

This notebook uses the same core setup as `fd_acoustic_1D.ipynb` from the Computational Geophysics short course: `nx=10000`, `nt=1001`, and `idisp=5`.

The purpose is to compare the same course-style animation on MyBinder and JupyterLite.

In [ ]:
import os
import sys
import time as timer
from pathlib import Path

for candidate in [Path('/files/notebooks'), Path('/notebooks'), Path.cwd(), Path.cwd() / 'notebooks']:
    if candidate.exists():
        os.chdir(candidate)
        sys.path.insert(0, str(candidate))
        print(f'working_directory={Path.cwd()}')
        break

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import gridspec
from IPython.display import HTML, display
from progress_bar import ProgressBarHandler
from animation_1d import create_animation_gaussian

plt.rcParams['animation.embed_limit'] = 250

In [ ]:
nx   = 10000
xmax = 10000
dx   = xmax / (nx - 1)
c0   = 334.
isrc = int(nx / 2)
nt   = 1001
dt   = 0.0010
f0   = 25.
t0   = 4. / f0
idisp = 5

src  = np.zeros(nt + 1)
time = np.linspace(0 * dt, nt * dt, nt)
src  = -8. * (time - t0) * f0 * (np.exp(-1.0 * (4 * f0) ** 2 * (time - t0) ** 2))

fig1 = plt.figure(figsize=(10, 6))
gs1  = gridspec.GridSpec(1, 2, width_ratios=[1, 1], hspace=0.3, wspace=0.3)
ax1  = plt.subplot(gs1[0])
ax1.plot(time, src)
ax1.set_title('Source Time Function')
ax1.set_xlim(time[0], time[-1])
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Amplitude')

ax2  = plt.subplot(gs1[1])
spec = np.fft.fft(src)
freq = np.fft.fftfreq(spec.size, d=dt)
ax2.plot(np.abs(freq), np.abs(spec))
ax2.set_xlim(0, 250)
ax2.set_title('Source Spectrum')
ax2.set_xlabel('Frequency (Hz)')
ax2.set_ylabel('Amplitude')
ax2.yaxis.tick_right()
ax2.yaxis.set_label_position('right')
plt.show()

In [ ]:
p    = np.zeros(nx)
pold = np.zeros(nx)
pnew = np.zeros(nx)
d2px = np.zeros(nx)
c    = np.zeros(nx) + c0
x    = np.arange(nx) * dx

fig = plt.figure(figsize=(10, 6))
gs  = gridspec.GridSpec(1, 1, width_ratios=[1], hspace=0.3, wspace=0.3)
ax  = plt.subplot(gs[0])
leg1, = ax.plot(isrc, 0, 'r*', markersize=11)
up31, = ax.plot(p)
ax.set_xlim(0, xmax)
ax.set_ylim(-np.max(p), np.max(p))
ax.set_title('Time Step (nt) = 0')
ax.set_xlabel('x (m)')
ax.set_ylabel('Pressure Amplitude')

p_results = np.zeros((math.ceil(nt / idisp), nx), dtype=np.float32)
window = 100
xshift = 25

start = timer.perf_counter()
progress_handler = ProgressBarHandler(nt, '1D Acoustic Wave Simulation')
for it in range(nt):
    for i in range(1, nx - 1):
        d2px[i] = (p[i + 1] - 2 * p[i] + p[i - 1]) / dx ** 2

    pnew = 2 * p - pold + c ** 2 * dt ** 2 * d2px
    pnew[isrc] = pnew[isrc] + src[it] / dx * dt ** 2
    pold, p = p, pnew

    if it % idisp == 0:
        p_results[it // idisp] = p

    progress_handler(it)

simulation_seconds = timer.perf_counter() - start
print(f'simulation_seconds={simulation_seconds:.3f}')
print(f'frames={len(p_results)}, samples_per_frame={p_results.shape[1]}')

In [ ]:
start = timer.perf_counter()
ani = create_animation_gaussian(locals())
animation_creation_seconds = timer.perf_counter() - start
print(f'animation_creation_seconds={animation_creation_seconds:.3f}')

start = timer.perf_counter()
html = ani.to_jshtml()
jshtml_render_seconds = timer.perf_counter() - start
print(f'jshtml_render_seconds={jshtml_render_seconds:.3f}')
print(f'html_size_mb={len(html) / 1024 / 1024:.2f}')
display(HTML(html))
plt.close()